# 07 – Métricas do Lote 6 (L6): validação da harmonização MODIS–VIIRS, por célula

HAR-01..HAR-07 segundo `spec/metodologia_metricas.md`, **por célula de grelha** (`cell`) em vez de por país, mais uma linha **global** (`cell = -1`) em cada tabela. Corre depois do notebook 02.

- **HAR-01/02** (sobreposição de células-dia e Jaccard por célula e mês) leem `processed/` (`modis_<ano>` e `viirs-jpss1_<ano>`, anos >= `CAL_START`) **um ano de cada vez**, na grelha `STATS_GRID_DEG`. A linha global é a soma das células (`global_overlap`).
- **HAR-03..HAR-07** leem `l1_cal_monthly` (esparsa) e `l1_meta_month_validity`, por **fatias de `ilat`** com cerca de `CHUNK_CELLS` células. Cada fatia densifica (`densify_monthly`) só as suas células. As tabelas são **esparsas** (só células com atividade desde `CAL_START`) e **quase todos os valores por célula são NA** (poucas contagens por mês, `MIN_PAIRS`/`MIN_CAL_MONTHS`): é o esperado.
- **Linha global:** com `include_global=False` em cada fatia, soma-se `comparable_pairs` de todas as fatias por (ano, mês) numa série global (`cell = -1`) e as funções correm **uma vez** sobre ela. O fator global (HAR-06) é depois o *fallback* das células sem fator próprio (`uses_global_fallback`). `l6_har06_factor` leva a linha global: os notebooks 08 e 09 precisam dela.
- Os parciais ficam em `metrics/_partial_lote6/` (um ano para HAR-01/02, uma fatia para HAR-03..07) com assinatura: **retoma** se a sessão cair (`FORCE = True` refaz). Consolida com DuckDB em `metrics/l6_*/` (um ficheiro Parquet ordenado por `ilat`, `ilon`; sem partição por `lat_band` porque as tabelas levam a linha global) e escreve `_runs` (lote `lote6`) **por último**.

O fator mapeia MODIS para a escala VIIRS pela razão das somas de calibração (2018–2021 por defeito; teste a partir de 2022). A gama *leave-one-year-out* e o erro-padrão *jackknife* descrevem a sensibilidade aos anos de calibração, não são um intervalo de confiança formal.

Não há poda por `MIN_ACTIVE_YEARS`: a harmonização tem os seus próprios mínimos. Use o ramo `metricas-pipeline`; para um repositório privado ative o segredo `GITHUB_TOKEN` do Colab. O ficheiro de um ano vazio é aceite.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64
import os
import subprocess
import sys

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Sprinklers')        # ajustar se o atalho tiver outro nome
PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
REPO = Path('/content/repo')
environment = os.environ.copy()
try:
    token = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    token = None
if token:
    credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    environment.update(GIT_CONFIG_COUNT='1',
                       GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                       GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {credential}')
    del credential
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', 'metricas-pipeline'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'metricas-pipeline'],
                   env=environment, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'metricas-pipeline',
                    'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git',
                    str(REPO)], env=environment, check=True)
del token, environment
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))

assert PROCESSED.exists(), f"Nao encontro {PROCESSED}. Corra primeiro o notebook 01 e confirme o atalho 'Sprinklers'."
METRICS.mkdir(exist_ok=True)


In [ ]:
# cell: params
YEARS = None        # None: todos os anos >= CAL_START com ficheiros processados dos dois sensores (HAR-01/02).
FORCE = False

# --- grelha e volume (iguais aos do notebook 02) ---
GRID_DEG = 0.01              # grelha das tabelas de base l1_*
STATS_GRID_DEG = GRID_DEG    # grelha das estatisticas (a de l1_stats_cal_monthly se for diferente)
CHUNK_CELLS = 20000          # celulas por fatia de HAR-03..07 (memoria: ~dezenas de milhares)

# --- periodos e regras NA da harmonizacao ---
CAL_START = 2018
CAL_END = 2021
TEST_START = 2022
MIN_PAIRS = 12
MIN_CAL_MONTHS = 24
MIN_CAL_YEARS = 3
MAX_UNKNOWN_DAYS = 0


In [ ]:
# cell: defs
import datetime, hashlib, importlib, json, re, subprocess, tempfile
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

from metrics import common, harmonization_factor, harmonization_overlap
from metrics.calendar_metrics import densify_monthly, eligible_cell_days
from metrics.store import ParquetSink, prepare_table, stats_source, write_run

for module in (harmonization_overlap, harmonization_factor):
    importlib.reload(module)

common._grid_shape(GRID_DEG)
common._grid_shape(STATS_GRID_DEG)
common.coarsen_cell(np.array([0]), GRID_DEG, STATS_GRID_DEG)     # ValueError se nao for multiplo
GLOBAL_CELL = common.GLOBAL_CELL

SINK = ParquetSink(METRICS)
PART = METRICS / '_partial_lote6'
PART.mkdir(exist_ok=True)
MONTHLY = stats_source('l1_cal_monthly', GRID_DEG, STATS_GRID_DEG)
VALIDITY = 'l1_meta_month_validity'
COMMIT = None
PERIOD_OPTIONS = dict(cal_start=CAL_START, cal_end=CAL_END, test_start=TEST_START, max_unknown_days=MAX_UNKNOWN_DAYS)
PARAMETERS = dict(PERIOD_OPTIONS, min_pairs=MIN_PAIRS, min_cal_months=MIN_CAL_MONTHS, min_cal_years=MIN_CAL_YEARS,
                  grid_deg=GRID_DEG, stats_grid_deg=STATS_GRID_DEG, chunk_cells=CHUNK_CELLS,
                  years=None if YEARS is None else sorted(YEARS))
PROCESSED_COLUMNS = ['instrument', 'satellite', 'latitude', 'longitude', 'acq_date', 'type', 'confidence_class']
FILE_RE = re.compile(r'(modis|viirs-jpss1)_([0-9]{4})')
NAMES = {'har01': 'l6_har01_overlap', 'har02': 'l6_har02_jaccard', 'har03': 'l6_har03_comparable_months',
         'har04': 'l6_har04_pearson', 'har05': 'l6_har05_spearman', 'har06': 'l6_har06_factor',
         'har07': 'l6_har07_factor_uncertainty'}
KEYS = {'har01': ['cell', 'year', 'month'], 'har02': ['cell', 'year', 'month'], 'har03': ['cell', 'period'],
        'har04': ['cell', 'period'], 'har05': ['cell', 'period'], 'har06': ['cell'], 'har07': ['cell']}
SORTS = {'har01': ['year', 'month'], 'har02': ['year', 'month'], 'har03': ['period'], 'har04': ['period'],
         'har05': ['period'], 'har06': [], 'har07': []}
SLICE_TABLES = ['har03', 'har04', 'har05', 'har06', 'har07']


def atomic_part(df, path):
    """Grava um parcial Parquet (colunas de texto sempre como `string`, mesmo sem valores)."""
    out = df.copy()
    for column in out.columns:
        if pd.api.types.is_object_dtype(out[column].dtype) or pd.api.types.is_string_dtype(out[column].dtype):
            out[column] = out[column].astype('string')
    tmp = path.with_name(path.name + '.tmp')
    out.to_parquet(tmp, index=False)
    tmp.replace(path)


def atomic_json(value, path):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(value, sort_keys=True), encoding='utf-8')
    tmp.replace(path)


def read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return None


def file_digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def tree_digest(name):
    """Assinatura de uma tabela publicada (pasta Parquet): ficheiros, tamanhos e mtimes."""
    root = SINK.path(name)
    items = sorted((p.relative_to(root).as_posix(), p.stat().st_size, p.stat().st_mtime_ns)
                   for p in root.rglob('*.parquet'))
    return hashlib.sha256(json.dumps(items).encode()).hexdigest()


def frame_digest(df):
    values = pd.util.hash_pandas_object(df.reset_index(drop=True), index=False).to_numpy().tobytes()
    return hashlib.sha256(values + ','.join(map(str, df.columns)).encode()).hexdigest()


def duckdb_connect():
    con = duckdb.connect()
    con.execute(f"SET temp_directory='{Path(tempfile.gettempdir()).as_posix()}/duckdb_spill'")
    return con


def sql_path(path):
    return str(path).replace('\\', '/').replace("'", "''")


def sql_list(files):
    return '[' + ', '.join("'" + sql_path(f) + "'" for f in files) + ']'


def table_source(name):
    """Fonte DuckDB que le a pasta Parquet publicada `name` (com as particoes hive)."""
    pattern = sql_path(SINK.path(name) / '**' / '*.parquet')
    return f"read_parquet('{pattern}', hive_partitioning=true, union_by_name=true)"


def scalar(name, expression):
    """Valor de uma agregacao SQL sobre a tabela publicada `name`."""
    con = duckdb_connect()
    try:
        value = con.execute(f'SELECT {expression} FROM {table_source(name)}').fetchone()[0]
    finally:
        con.close()
    return 0 if value is None else value


def consolidate(name, files, order_by, keys):
    """Junta os parciais `files` na tabela publicada `name` (um ficheiro Parquet ordenado por `order_by`,
    com DuckDB fora da memoria). Verifica chaves repetidas e devolve o numero de linhas.
    Nao se particiona por `lat_band`: estas tabelas levam linhas globais (cell = -1, sem lat_band)."""
    files = [Path(f) for f in files]
    source = f'read_parquet({sql_list(files)}, union_by_name=true)'
    con = duckdb_connect()
    try:
        n_rows = con.execute(f'SELECT count(*) FROM {source}').fetchone()[0]
        if n_rows:
            repeated = con.execute(
                f'SELECT count(*) FROM (SELECT 1 FROM {source} GROUP BY {", ".join(keys)} HAVING count(*) > 1)'
            ).fetchone()[0]
            assert repeated == 0, f'{name}: {repeated} chaves repetidas entre parciais'
    finally:
        con.close()
    if n_rows == 0:
        SINK.write(name, pd.read_parquet(files[0]).iloc[:0], ())
        return 0
    SINK.write_from_parquet(name, files, order_by=order_by, partition_cols=())
    return int(n_rows)


def plain(df):
    """Colunas de particao (hive) voltam como categoricas: passa-as a tipos normais."""
    for column in df.columns:
        if isinstance(df[column].dtype, pd.CategoricalDtype):
            df[column] = df[column].astype(df[column].cat.categories.dtype)
    return df


def git_commit():
    try:
        return subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True,
                              text=True).stdout.strip() or 'unknown'
    except Exception:
        return 'unknown'


def plan_slices():
    """[(ilat_lo, ilat_hi)]: faixas de ilat contiguas com cerca de CHUNK_CELLS celulas ativas (series
    modis/viirs, ano >= CAL_START) cada uma. Uma faixa nunca parte um ilat; as celulas de uma faixa maior
    que CHUNK_CELLS sao depois processadas em blocos de CHUNK_CELLS."""
    con = duckdb_connect()
    try:
        rows = con.execute(
            f"SELECT ilat, count(DISTINCT cell) FROM {table_source(MONTHLY)} "
            f"WHERE series IN ('modis', 'viirs') AND year >= {int(CAL_START)} GROUP BY ilat ORDER BY ilat"
        ).fetchall()
    finally:
        con.close()
    plan, lo, last, total = [], None, None, 0
    for ilat, n in rows:
        if lo is not None and total + n > CHUNK_CELLS:
            plan.append((lo, last))
            lo, total = None, 0
        if lo is None:
            lo = int(ilat)
        last, total = int(ilat), total + int(n)
    if lo is not None:
        plan.append((lo, last))
    return plan


def read_slice(lo, hi):
    """Linhas esparsas de l1_cal_monthly (so modis/viirs, ano >= CAL_START) com ilat em [lo, hi]."""
    band_lo, band_hi = (int(common.lat_band([ilat], STATS_GRID_DEG)[0]) for ilat in (lo, hi))
    filters = [('lat_band', '>=', band_lo), ('lat_band', '<=', band_hi), ('ilat', '>=', lo), ('ilat', '<=', hi),
               ('year', '>=', int(CAL_START))]
    sparse = plain(pd.read_parquet(SINK.path(MONTHLY), columns=['cell', 'year', 'month', 'series', 'cell_days'],
                                   filters=filters))
    sparse = sparse[sparse['series'].isin(['modis', 'viirs'])].copy()
    sparse['series'] = sparse['series'].astype(object)
    sparse['cell'] = sparse['cell'].astype('int64')
    return sparse.reset_index(drop=True)


def read_validity():
    validity = SINK.read(VALIDITY)
    validity = validity[validity['series'].isin(['modis', 'viirs']) & (validity['year'] >= CAL_START)].copy()
    validity['series'] = validity['series'].astype(object)
    return validity.reset_index(drop=True)


def apply_fallback(factor, global_factor):
    """Mesma regra de har06_factor: a celula sem fator proprio usa o fator global (se existir)."""
    out = factor.copy()
    if pd.notna(global_factor):
        missing = out['factor'].isna() & out['cell'].ne(GLOBAL_CELL)
        out.loc[missing, 'factor_used'] = global_factor
        out.loc[missing, 'uses_global_fallback'] = True
    return out


def compute_slice(lo, hi, directory, validity):
    """HAR-03..07 das celulas com ilat em [lo, hi], em blocos de CHUNK_CELLS (sem linha global)."""
    sparse = read_slice(lo, hi)
    cells = np.sort(sparse['cell'].unique())
    results = {name: [] for name in SLICE_TABLES}
    sums = []
    for start in range(0, len(cells), CHUNK_CELLS):
        block = cells[start:start + CHUNK_CELLS]
        dense = densify_monthly(sparse[sparse['cell'].isin(block)], validity, cells=block)
        options = dict(include_global=False, **PERIOD_OPTIONS)
        pairs = harmonization_factor.comparable_pairs(dense, **options)
        sums.append(pairs.groupby(['year', 'month'], as_index=False)[['modis', 'viirs']].sum())
        results['har03'].append(harmonization_factor.har03_comparable_months(dense, **options))
        results['har04'].append(harmonization_factor.har04_pearson(dense, min_pairs=MIN_PAIRS, **options))
        results['har05'].append(harmonization_factor.har05_spearman(dense, min_pairs=MIN_PAIRS, **options))
        results['har06'].append(harmonization_factor.har06_factor(dense, min_cal_months=MIN_CAL_MONTHS, **options))
        results['har07'].append(harmonization_factor.har07_factor_uncertainty(
            dense, min_cal_months=MIN_CAL_MONTHS, min_cal_years=MIN_CAL_YEARS, **options))
        del dense
    rows = {}
    for name in SLICE_TABLES:
        table = pd.concat(results[name], ignore_index=True)
        atomic_part(prepare_table(table, STATS_GRID_DEG, SORTS[name]), directory / f'{name}.parquet')
        rows[name] = len(table)
    total = pd.concat(sums, ignore_index=True).groupby(['year', 'month'], as_index=False)[['modis', 'viirs']].sum()
    atomic_part(total, directory / 'pairs.parquet')
    return rows


In [ ]:
# cell: inputs
COMMIT = git_commit()
missing = [str(SINK.path(name)) for name in (MONTHLY, VALIDITY) if not SINK.exists(name)]
if missing:
    raise FileNotFoundError('Missing L1 inputs: ' + ', '.join(missing) + '. Run 02_metrics_lote1.ipynb first.')
if YEARS is None:
    years = set()
    for path in PROCESSED.glob('*.parquet'):
        match = FILE_RE.fullmatch(path.stem)
        if match and int(match[2]) >= CAL_START:
            years.add(int(match[2]))
    years = sorted(years)
else:
    years = sorted({int(year) for year in YEARS if int(year) >= CAL_START})
if not years:
    raise FileNotFoundError(f'No selected processed sensor-year files in {PROCESSED}')
sources = {year: [PROCESSED / f'{sensor}_{year}.parquet' for sensor in ['modis', 'viirs-jpss1']] for year in years}
missing = [str(path) for paths in sources.values() for path in paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing processed counterparts: ' + ', '.join(missing))
overlap_digest = file_digest(harmonization_overlap.__file__)
factor_digest = file_digest(harmonization_factor.__file__)
validity = read_validity()
print(f'{len(years)} anos para HAR-01/02: {years}')


In [ ]:
# cell: overlap
overlap_files, jaccard_files = [], []
for year in years:
    overlap_part = PART / f'har01_{year}.parquet'
    jaccard_part = PART / f'har02_{year}.parquet'
    marker = PART / f'har01_{year}.json'
    signature = dict(schema=1, cal_start=CAL_START, stats_grid_deg=STATS_GRID_DEG, code=overlap_digest,
                     commit=COMMIT, sources=[dict(name=path.name, size=path.stat().st_size,
                                                  mtime_ns=path.stat().st_mtime_ns) for path in sources[year]])
    overlap_files.append(overlap_part)
    jaccard_files.append(jaccard_part)
    if not FORCE and overlap_part.is_file() and jaccard_part.is_file() and read_json(marker) == signature:
        print(f'Reusing completed overlap for {year}')
        continue
    marker.unlink(missing_ok=True)
    compact = []
    for instrument, path in zip(['MODIS', 'VIIRS'], sources[year]):
        frame = pd.read_parquet(path, columns=PROCESSED_COLUMNS)
        if not frame.instrument.astype('string').str.upper().eq(instrument).fillna(False).all():
            raise ValueError(f'Unexpected instrument in {path}')
        for column in ['instrument', 'satellite', 'confidence_class']:
            frame[column] = frame[column].astype('category')
        cell_days = eligible_cell_days(frame, grid_deg=STATS_GRID_DEG)
        if not cell_days.date.dt.year.eq(year).all():
            raise ValueError(f'Acquisition dates outside filename year in {path}')
        compact.append(cell_days)
        del frame
    cell_days = pd.concat(compact, ignore_index=True)
    overlap = harmonization_overlap.har01_cellday_overlap(cell_days, cal_start=CAL_START)
    overlap = pd.concat([overlap, harmonization_overlap.global_overlap(overlap)], ignore_index=True)
    jaccard = harmonization_overlap.har02_jaccard(overlap)
    atomic_part(prepare_table(overlap, STATS_GRID_DEG, SORTS['har01']), overlap_part)
    atomic_part(prepare_table(jaccard, STATS_GRID_DEG, SORTS['har02']), jaccard_part)
    atomic_json(signature, marker)
    del compact, cell_days
    print(f'Processed overlap for {year}')


In [ ]:
# cell: slices
monthly_digest, validity_digest = tree_digest(MONTHLY), frame_digest(validity)
plan = plan_slices()
slice_dirs = []
for lo, hi in plan:
    directory = PART / f'har_{lo}_{hi}'
    marker = directory / '_done.json'
    signature = json.loads(json.dumps(dict(
        schema=1, ilat=[lo, hi], chunk_cells=CHUNK_CELLS, stats_grid_deg=STATS_GRID_DEG, period=PERIOD_OPTIONS,
        min_pairs=MIN_PAIRS, min_cal_months=MIN_CAL_MONTHS, min_cal_years=MIN_CAL_YEARS,
        code=factor_digest, monthly=monthly_digest, validity=validity_digest)))
    slice_dirs.append(directory)
    info = read_json(marker)
    if (not FORCE and isinstance(info, dict) and info.get('signature') == signature
            and all((directory / f'{name}.parquet').is_file() for name in SLICE_TABLES + ['pairs'])):
        print(f'Reusing slice ilat {lo}..{hi}')
        continue
    directory.mkdir(exist_ok=True)
    marker.unlink(missing_ok=True)
    rows = compute_slice(lo, hi, directory, validity)
    atomic_json(dict(signature=signature, rows=rows), marker)
    print(f'Processed slice ilat {lo}..{hi}: {rows}')
print(f'{len(slice_dirs)} fatias de HAR-03..07')


In [ ]:
# cell: global
# serie global (cell = -1): soma, por (ano, mes), dos pares comparaveis de todas as fatias
pair_parts = [pd.read_parquet(directory / 'pairs.parquet') for directory in slice_dirs]
empty_pairs = pd.DataFrame({column: pd.Series([], dtype='int64') for column in ['year', 'month', 'modis', 'viirs']})
pair_sums = pd.concat(pair_parts + [empty_pairs], ignore_index=True).groupby(['year', 'month'], as_index=False)[
    ['modis', 'viirs']].sum()
global_monthly = pd.concat([pair_sums.assign(series=series, cell_days=pair_sums[series])
                            for series in ('modis', 'viirs')], ignore_index=True)
global_monthly = global_monthly[['year', 'month', 'series', 'cell_days']].assign(
    cell=GLOBAL_CELL, partial=False, n_unknown_days=0)
options = dict(include_global=False, **PERIOD_OPTIONS)
global_tables = {
    'har03': harmonization_factor.har03_comparable_months(global_monthly, **options),
    'har04': harmonization_factor.har04_pearson(global_monthly, min_pairs=MIN_PAIRS, **options),
    'har05': harmonization_factor.har05_spearman(global_monthly, min_pairs=MIN_PAIRS, **options),
    'har06': harmonization_factor.har06_factor(global_monthly, min_cal_months=MIN_CAL_MONTHS, **options),
    'har07': harmonization_factor.har07_factor_uncertainty(
        global_monthly, min_cal_months=MIN_CAL_MONTHS, min_cal_years=MIN_CAL_YEARS, **options),
}
GLOBAL_DIR = PART / 'global'
GLOBAL_DIR.mkdir(exist_ok=True)
for name, table in global_tables.items():
    atomic_part(prepare_table(table, STATS_GRID_DEG, SORTS[name]), GLOBAL_DIR / f'{name}.parquet')
global_factor = global_tables['har06'].loc[global_tables['har06'].cell.eq(GLOBAL_CELL), 'factor']
global_factor = float(global_factor.iloc[0]) if len(global_factor) else float('nan')
print('Fator global (HAR-06):', global_factor)


In [ ]:
# cell: publish
# fallback global nas celulas sem fator proprio (o fator global so se conhece depois de todas as fatias)
for directory in slice_dirs:
    final = apply_fallback(pd.read_parquet(directory / 'har06.parquet'), global_factor)
    atomic_part(final, directory / 'har06_final.parquet')

files = {'har01': overlap_files, 'har02': jaccard_files}
for name in SLICE_TABLES:
    stem = 'har06_final' if name == 'har06' else name
    files[name] = [directory / f'{stem}.parquet' for directory in slice_dirs] + [GLOBAL_DIR / f'{name}.parquet']
published = {}
for key, name in NAMES.items():
    published[name] = consolidate(name, files[key], ['ilat', 'ilon'] + SORTS[key], KEYS[key])
    print(f'{name}: {published[name]} linhas')

# _runs escreve-se POR ULTIMO: uma execucao que parou a meio nao fica registada
write_run(SINK, 'lote6', commit=COMMIT, grid_deg=STATS_GRID_DEG, tables=published, params=dict(
    PARAMETERS, overlap_years=years, n_slices=len(slice_dirs), global_factor=global_factor,
    sources=[path.name for paths in sources.values() for path in paths],
    run_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')))


In [ ]:
# cell: verify
O1, O6 = NAMES['har01'], NAMES['har06']
assert scalar(O1, 'count(*) FILTER (WHERE n_union <> n_modis + n_viirs - n_both)') == 0, 'HAR-01: n_union incoerente'
assert scalar(NAMES['har02'], 'count(*) FILTER (WHERE jaccard < 0 OR jaccard > 1)') == 0, 'HAR-02: Jaccard fora de [0, 1]'
con = duckdb_connect()
try:
    for column in ['n_modis', 'n_viirs', 'n_both', 'n_union']:     # a linha global soma as celulas, mes a mes
        bad = con.execute(
            f"SELECT count(*) FROM (SELECT year, month, sum({column}) FILTER (WHERE cell <> {GLOBAL_CELL}) AS cells, "
            f"sum({column}) FILTER (WHERE cell = {GLOBAL_CELL}) AS total FROM {table_source(O1)} "
            "GROUP BY year, month HAVING cells IS DISTINCT FROM total)").fetchone()[0]
        assert bad == 0, f'HAR-01: linha global != soma das celulas ({column})'
finally:
    con.close()
n_global = scalar(O6, f'count(*) FILTER (WHERE cell = {GLOBAL_CELL})')
assert n_global == (1 if len(global_monthly) else 0), 'HAR-06: falta (ou repete-se) a linha global'
for column in ['sum_modis', 'sum_viirs']:
    cells = scalar(O6, f'coalesce(sum({column}) FILTER (WHERE cell <> {GLOBAL_CELL}), 0)')
    total = scalar(O6, f'coalesce(sum({column}) FILTER (WHERE cell = {GLOBAL_CELL}), 0)')
    assert cells == total, f'HAR-06: global != soma das celulas ({column})'
if global_factor == global_factor:      # fator global definido: as celulas sem fator proprio usam-no
    bad = scalar(O6, f"count(*) FILTER (WHERE factor IS NULL AND cell <> {GLOBAL_CELL} AND "
                     f"(factor_used IS DISTINCT FROM CAST({global_factor!r} AS DOUBLE) OR NOT uses_global_fallback))")
    assert bad == 0, 'HAR-06: fallback global mal aplicado'
print('OK: invariantes verificadas')
print(published)
